# Fuel Demand Forecasting — Full Pipeline Notebook (v5)
**Cofano / ROVER**

This notebook presents the complete research workflow, from exploratory data analysis through model
training and evaluation, for forecasting fuel demand at the **(station × product × day)** level. The
business objective is to supply forecasts to the ROVER optimiser so as to reduce the *levels penalty*
(the cost of incorrect inventory levels). Modelling decisions are driven by evidence in the data, and the notebook is self-contained for Colab execution using the required uploaded CSV files. Version 5 adds an explicit, auditable distinction between confirmed zero demand, unknown whole-station days, and missing historical features.

**Handover note:** repository outputs were regenerated from this Version 5 code and saved in `outputs/`. Embedded cell outputs are cleared before handover so no Version 4 result can be mistaken for a Version 5 result; use **Runtime → Run all** in Colab to embed the tables and visualisations again.

**Outline:** (0) Setup · (1) Data overview & quality · (2) Exploratory analysis (EDA) ·
(3) Leakage-safe feature engineering · (4) Backtest design · (5) Feature-group ablation (5 folds) ·
(6) Statistical testing (bootstrap) · (7) Error by segment · (8) Quantile (safety-stock) layer ·
(9) Levels-penalty simulation · (10) Conclusions.

In [ ]:
# Colab / clean-environment safety: reg:quantileerror (Section 8) needs xgboost >= 2.0.
!pip install -q "xgboost>=2.0.0"

## 0. Setup & data loading

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize":(11,4),"axes.grid":True,"grid.alpha":.3,
                     "axes.spines.top":False,"axes.spines.right":False,"font.size":10})

# =========================================================
# COLAB INPUT CHECK
# =========================================================
DATASET_FILE = "demand_history.csv"
GEO_FILE = "station_metadata.csv"

print("COLAB INPUT REQUIRED - please upload these CSV files before running all cells:")
print(f"1. {DATASET_FILE}")
print(f"2. {GEO_FILE}")
# NOTE: sim_penalty.csv is no longer required -- Section 9 now runs its own capacity-aware
# simulation instead of reading a precomputed penalty table (see Section 9 for why).
print("You can upload them directly to /content or place them in a folder named colab_upload_csv.")

candidate_dirs = [
    Path("/content"),
    Path("/content/colab_upload_csv"),
    Path.cwd(),
    Path.cwd() / "colab_upload_csv",
    Path.cwd() / "data",
]

def find_required_csv(file_name):
    for folder in candidate_dirs:
        candidate = folder / file_name
        if candidate.exists():
            return candidate
    searched = "\n".join(str(p / file_name) for p in candidate_dirs)
    raise FileNotFoundError(
        f"Missing required CSV file: {file_name}\n"
        f"Please upload it before running this notebook.\nSearched:\n{searched}"
    )

DATASET_PATH = find_required_csv(DATASET_FILE)
GEO_PATH = find_required_csv(GEO_FILE)
OUTPUT_DIR = Path("/content/outputs") if Path("/content").exists() else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nCSV input files found:")
print("- Main dataset:", DATASET_PATH)
print("- Station metadata:", GEO_PATH)
print("- Output CSV files will be saved to:", OUTPUT_DIR)

# =========================================================
# PIPELINE CONFIGURATION USED BY THIS NOTEBOOK
# =========================================================
TARGET = "daily_demand"
PRODUCT_FUEL = {"P1":"Euro95", "P2":"Super98", "P3":"Diesel", "P4":"AdBlue", "P5":"LPG"}
MAIN_PRODUCTS = ["P1", "P3"]

LAGS = [7, 14, 21, 28, 56, 364, 371]
G_TS = ["lag_7","lag_14","lag_21","lag_28","lag_56","lag_364","lag_371",
        "roll4_mean","roll4_std","roll8_mean","roll8_std","asof_last","yoy_mean","trend_7_28","ratio_7_28",
        "lag_7_missing","lag_14_missing","lag_21_missing","lag_28_missing","lag_56_missing",
        "lag_364_missing","lag_371_missing","roll4_valid_count","roll8_valid_count"]
G_CAL = ["day_of_week","is_weekend","month","quarter","week_of_year","day_of_year",
         "month_sin","month_cos","week_sin","week_cos","is_month_end"]
G_STAT = ["station_encoded","product_encoded","country_encoded","area_type_encoded","nearest_place_type_encoded",
          "traffic_source_encoded","station_product_encoded","lat","lon","building_density_km2","roads_1km",
          "major_roads_3km","is_belgium"]
G_SCHOOL = ["is_school_holiday","school_holiday_type_code","days_into_holiday","days_to_holiday"]
BEST_FEATURES = G_TS + G_CAL + G_STAT + G_SCHOOL
SEED = 0
QUANTILES = [0.50, 0.90, 0.95]

ENSEMBLE_CFGS = [
    dict(n_estimators=300, learning_rate=0.05, max_depth=7, subsample=0.9, colsample_bytree=0.9,
         objective="reg:tweedie", tweedie_variance_power=1.4, tree_method="hist", n_jobs=-1, eval_metric="mae", early_stopping_rounds=30),
    dict(n_estimators=300, learning_rate=0.03, max_depth=7, subsample=0.9, colsample_bytree=0.9,
         objective="reg:absoluteerror", tree_method="hist", n_jobs=-1, eval_metric="mae", early_stopping_rounds=30),
]
QUANTILE_PARAMS = dict(n_estimators=300, learning_rate=0.04, max_depth=5, subsample=0.9, colsample_bytree=0.9,
                       objective="reg:quantileerror", tree_method="hist", n_jobs=-1, eval_metric="mae", early_stopping_rounds=30)

def _encode(series):
    return pd.factorize(series.astype(str).fillna("missing"))[0]

def build_daily_panel(raw):
    """Build a calendar-day panel without silently equating data outages with zero demand.

    Policy: an observed row keeps its actual target; a missing product row is a confirmed zero only
    when another product was observed at the same station on that date; if the whole station-day is
    absent, the target remains unknown. Pairs are expanded only between their first and last observed
    dates, avoiding invented history before a product launch or after discontinuation.
    """
    base = raw.copy()
    base["date"] = pd.to_datetime(base["date"], errors="coerce")
    if TARGET not in base.columns and "volume_in_liter" in base.columns:
        base[TARGET] = base["volume_in_liter"]
    base = base.dropna(subset=["date", "station_code", "product"]).copy()
    duplicate_keys = base.duplicated(["station_code", "product", "date"], keep=False)
    if duplicate_keys.any():
        raise ValueError(f"Expected one row per station-product-day; found {int(duplicate_keys.sum())} duplicate rows.")

    spans = base.groupby(["station_code", "product"])["date"].agg(["min", "max"]).reset_index()
    panel_parts = [pd.DataFrame({"station_code": r["station_code"], "product": r["product"],
                                 "date": pd.date_range(r["min"], r["max"], freq="D")})
                   for _, r in spans.iterrows()]
    panel = pd.concat(panel_parts, ignore_index=True)
    panel = panel.merge(base, on=["station_code", "product", "date"], how="left", indicator="_source")
    panel["source_observed"] = panel["_source"].eq("both")
    panel = panel.drop(columns="_source")

    station_day = base[["station_code", "date"]].drop_duplicates().assign(station_day_observed=True)
    panel = panel.merge(station_day, on=["station_code", "date"], how="left")
    panel["station_day_observed"] = panel["station_day_observed"].fillna(False).astype(bool)
    panel["target_status"] = np.select(
        [panel["source_observed"], panel["station_day_observed"]],
        ["observed", "confirmed_zero"], default="unknown_station_day")
    panel.loc[panel["target_status"].eq("confirmed_zero"), TARGET] = 0.0
    panel["target_known"] = (~panel["target_status"].eq("unknown_station_day")) & panel[TARGET].notna()

    # A peer product supplies same-station/day metadata for confirmed-zero rows. Static fields are
    # then propagated within each active pair; target and audit columns are never propagated.
    audit_cols = {"station_code", "product", "date", TARGET, "source_observed",
                  "station_day_observed", "target_status", "target_known"}
    metadata_cols = [c for c in base.columns if c not in audit_cols]
    station_day_meta = base.groupby(["station_code", "date"], as_index=False)[metadata_cols].first()
    panel = panel.merge(station_day_meta, on=["station_code", "date"], how="left", suffixes=("", "_station_day"))
    for col in metadata_cols:
        peer = f"{col}_station_day"
        if peer in panel.columns:
            panel[col] = panel[col].combine_first(panel[peer])
            panel = panel.drop(columns=peer)
        panel[col] = panel.groupby(["station_code", "product"])[col].transform(lambda s: s.ffill().bfill())

    panel["year_month"] = panel["date"].dt.to_period("M").astype(str)
    observed_days = (station_day.assign(year_month=station_day["date"].dt.to_period("M").astype(str))
                     .groupby(["station_code", "year_month"])["date"].nunique()
                     .rename("station_month_observed_days").reset_index())
    panel = panel.merge(observed_days, on=["station_code", "year_month"], how="left")
    panel["station_month_observed_days"] = panel["station_month_observed_days"].fillna(0).astype(int)
    panel["month_eligible"] = panel["station_month_observed_days"].ge(28)
    return panel.sort_values(["station_code", "product", "date"]).reset_index(drop=True)

def build_features(raw):
    df = build_daily_panel(raw)
    df = df.sort_values(["station_code", "product", "date"]).reset_index(drop=True)
    keys = ["station_code", "product"]

    # Calendar features
    df["day_of_week"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["day_of_year"] = df["date"].dt.dayofyear
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
    df["week_sin"] = np.sin(2*np.pi*df["week_of_year"]/52)
    df["week_cos"] = np.cos(2*np.pi*df["week_of_year"]/52)
    df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

    # These columns are already included in the prepared Colab CSV. Defaults are safety fallbacks.
    for col, default in [("is_school_holiday",0),("school_holiday_type_code",0),("days_into_holiday",-1),("days_to_holiday",-1)]:
        if col not in df.columns:
            df[col] = default

    # Price aliases for the price ablation group.
    for old, new in [("fuelprice_euro95_eur_l","price_euro95"),("fuelprice_diesel_eur_l","price_diesel"),("fuelprice_lpg_eur_l","price_lpg")]:
        if old in df.columns and new not in df.columns:
            df[new] = pd.to_numeric(df[old], errors="coerce")

    # Time-series features: shifted by at least 7 days for weekly planning.
    grouped = df.groupby(keys)[TARGET]
    for lag in LAGS:
        df[f"lag_{lag}"] = grouped.shift(lag)
        df[f"lag_{lag}_missing"] = df[f"lag_{lag}"].isna().astype(int)
    df["asof_last"] = df.groupby(keys)[TARGET].transform(lambda s: s.shift(7).ffill())
    df["roll4_mean"] = df.groupby(keys)[TARGET].transform(lambda s: s.shift(7).rolling(28, min_periods=1).mean())
    df["roll4_std"] = df.groupby(keys)[TARGET].transform(lambda s: s.shift(7).rolling(28, min_periods=2).std())
    df["roll8_mean"] = df.groupby(keys)[TARGET].transform(lambda s: s.shift(7).rolling(56, min_periods=1).mean())
    df["roll8_std"] = df.groupby(keys)[TARGET].transform(lambda s: s.shift(7).rolling(56, min_periods=2).std())
    df["roll4_valid_count"] = df.groupby(keys)[TARGET].transform(lambda s: s.shift(7).rolling(28, min_periods=1).count())
    df["roll8_valid_count"] = df.groupby(keys)[TARGET].transform(lambda s: s.shift(7).rolling(56, min_periods=1).count())
    df["yoy_mean"] = df[["lag_364", "lag_371"]].mean(axis=1)
    df["trend_7_28"] = df["lag_7"] - df["lag_28"]
    df["ratio_7_28"] = df["lag_7"] / df["lag_28"].replace(0, np.nan)

    # Station / geography features.
    df["station_encoded"] = _encode(df["station_code"])
    df["product_encoded"] = _encode(df["product"])
    df["country_encoded"] = _encode(df.get("country_code", pd.Series("missing", index=df.index)))
    df["area_type_encoded"] = _encode(df.get("area_type", pd.Series("missing", index=df.index)))
    df["nearest_place_type_encoded"] = _encode(df.get("nearest_place_type", pd.Series("missing", index=df.index)))
    df["traffic_source_encoded"] = _encode(df.get("traffic_source", pd.Series("missing", index=df.index)))
    df["station_product_encoded"] = _encode(df["station_code"].astype(str) + "_" + df["product"].astype(str))
    df["is_belgium"] = (df.get("country_code", "") == "BE").astype(int) if "country_code" in df.columns else 0
    for col in ["lat","lon","building_density_km2","roads_1km","major_roads_3km"]:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in BEST_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df, {}

import xgboost as xgb
raw = pd.read_csv(DATASET_PATH)
raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
geo = pd.read_csv(GEO_PATH)
print("\nRaw shape:", raw.shape, "| range", raw.date.min().date(), "->", raw.date.max().date())
raw.head(3)

## 1. Data overview & quality
Before modelling we examine the structure, coverage and integrity of the data. This is a necessary
step: forecast quality is upper-bounded by the quality of the input data.

In [ ]:
print("Stations               :", raw.station_code.nunique())
print("Products               :", sorted(raw['product'].unique()), "->", PRODUCT_FUEL)
print("Station-product series :", raw.groupby(['station_code','product']).ngroups, "(of a possible 20x5=100)")
print("Country split          :", raw.country_code.value_counts().to_dict())
print("Duplicate ids          :", int(raw.id.duplicated().sum()))
slen = raw.groupby(['station_code','product']).size()
print("Series length (min/median/max):", slen.min(), int(slen.median()), slen.max())
miss = (raw.isna().mean()*100).round(2); miss = miss[miss>0].sort_values(ascending=False)
print("\n% missing (columns with any missing):"); print(miss.to_string())

In [ ]:
desc = raw.groupby("product")[TARGET].describe()[["mean","50%","std","min","max"]].round(1)
desc.insert(0,"fuel",[PRODUCT_FUEL[p] for p in desc.index]); desc

**Reading the numbers above:** the 20 stations cover 5 products, but only 69 of the 100 possible station-product combinations are actually sold (not every station carries every fuel). There are no duplicate transaction ids and no unexpected structural gaps, so the daily panel can be built directly from these records without a separate cleaning pass. The per-product summary also previews the scale gap that shapes every later modelling choice: Euro95 and Diesel sell in the thousands of litres/day while AdBlue and LPG sell in the tens/hundreds — a single global model must lean on station/product identity features (Section 3) rather than one flat scale.

## 2. Exploratory data analysis (EDA)

### 2.1 Network-wide demand over time

In [ ]:
daily = raw.groupby("date")[TARGET].sum()
ax = daily.rolling(7).mean().plot(color="#0a9c4a", lw=1.7, label="7-day moving average")
daily.plot(color="#cdd8e6", lw=.6, alpha=.7, ax=ax, label="daily")
ax.set(title="Network-wide demand (litres/day)", ylabel="litres/day"); ax.legend(); plt.show()

Network-wide demand is clearly **noisy week to week but mean-reverting**: the 7-day moving average removes most of the daily noise and exposes a repeating weekly rhythm rather than a long-run trend break. This is the first visual evidence that lag/rolling features anchored to the week (Section 3) should carry most of the predictive signal, more so than a long-horizon trend term.

### 2.2 Distribution of the target

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(11,3.6))
raw[TARGET].clip(upper=raw[TARGET].quantile(.99)).hist(bins=60, ax=ax[0], color="#2563eb")
ax[0].set(title="daily_demand (clipped at p99)", xlabel="litres/day")
np.log1p(raw[TARGET]).hist(bins=60, ax=ax[1], color="#7c4dff")
ax[1].set(title="log1p(daily_demand)", xlabel="log litres"); plt.tight_layout(); plt.show()
print("Skewness:", round(raw[TARGET].skew(),2), "| mean/median:", round(raw[TARGET].mean(),1), "/", round(raw[TARGET].median(),1))

The **positive skew and mean above median** confirm the right tail seen in Report 1: a handful of high-volume station-product-days pull the average up. This is why WAPE (which weights every litre equally) is used as the primary metric instead of a plain percentage error, and why the ensemble later combines a Tweedie objective (built for skewed, non-negative targets) with an absolute-error objective rather than a plain squared-error model.

### 2.3 Volume composition by product

In [ ]:
g = raw.groupby("product")[TARGET].agg(["sum","mean"]); g["fuel"]=[PRODUCT_FUEL[p] for p in g.index]
g["share_%"]=(g["sum"]/g["sum"].sum()*100).round(2); g=g.sort_values("sum",ascending=False)
fig, ax = plt.subplots(1,2,figsize=(11,3.6))
ax[0].bar([f"{p}\n{g.loc[p,'fuel']}" for p in g.index], g["share_%"], color="#0a9c4a")
for i,(p,r) in enumerate(g.iterrows()): ax[0].text(i,r["share_%"]+.5,f"{r['share_%']}%",ha="center",fontsize=8)
ax[0].set(title="Share of observed demand volume", ylabel="%")
ax[1].bar([f"{p}\n{g.loc[p,'fuel']}" for p in g.index], g["mean"], color="#2563eb"); ax[1].set(title="Mean litres per observed station-product-day")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "product_volume_profile.png", dpi=180, bbox_inches="tight"); plt.show()
print("FIGURE SAVED:", OUTPUT_DIR / "product_volume_profile.png")
print("Euro95 + Diesel =", round(g.loc[['P1','P3'],'share_%'].sum(),1),"% of volume; AdBlue+LPG =", round(g.loc[['P4','P5'],'share_%'].sum(),2),"%")

Euro95 and Diesel together account for the large majority of observed demand volume. Section 7 therefore reports a **main-fuel WAPE** (Euro95 + Diesel) alongside the all-product result. This is a volume-based reporting choice: it does not assume that the dataset measures truck loading, delivery impact, or actual replenishment decisions.

### 2.4 Volume by station

In [ ]:
raw.groupby("station_code")[TARGET].mean().sort_values(ascending=False).plot.bar(color="#0a9c4a",figsize=(11,3.3))
plt.title("Mean demand (litres/day) by station"); plt.ylabel("litres/day"); plt.tight_layout(); plt.show()

One station (B2) dominates — a large highway site — creating substantial between-series variance. This heterogeneity is why the model pools all stations into **one global model** (Section 3) rather than training up to 100 independent station-product models: most series are too short or sparse to support an independent model, and a shared model with station/geography features lets low-volume stations borrow strength from the network.

### 2.5 Weekly and monthly seasonality

In [ ]:
raw["dow"]=raw.date.dt.dayofweek; raw["mon"]=raw.date.dt.month
fig,ax=plt.subplots(1,2,figsize=(11,3.3))
raw.groupby("dow")[TARGET].mean().plot(marker="o",color="#e11d65",ax=ax[0])
ax[0].set(title="By day of week",xlabel="0=Mon..6=Sun",ylabel="mean litres/day"); ax[0].set_xticks(range(7))
ax[0].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
raw.groupby("mon")[TARGET].mean().plot(marker="o",color="#e8820c",ax=ax[1]); ax[1].set(title="By month",xlabel="month"); ax[1].set_xticks(range(1,13))
plt.tight_layout(); plt.show()

The **weekday pattern is the strongest and most stable signal in the dataset**: demand drops sharply on Saturday/Sunday and stays flat across weekdays, matching a commuting/refuelling pattern rather than a leisure pattern. The monthly view is noisier and should be read cautiously given roughly two years of history. Together, these two panels justify keeping `day_of_week` (raw + sin/cos) as a core calendar feature while treating month/season as a secondary signal.

### 2.6 Annual seasonality: year-over-year

In [ ]:
ts = raw.groupby("date")[TARGET].sum()
yoy = pd.DataFrame({"2024":ts[ts.index.year==2024].groupby(ts[ts.index.year==2024].index.dayofyear).mean(),
                    "2025":ts[ts.index.year==2025].groupby(ts[ts.index.year==2025].index.dayofyear).mean()})
yoy.rolling(7).mean().plot(figsize=(11,3.3)); plt.title("Network demand by day-of-year (7d smoothed): 2024 vs 2025")
plt.xlabel("day of year"); plt.ylabel("litres/day"); plt.tight_layout(); plt.show()
print("Correlation 2024 vs 2025 (by day-of-year):", round(yoy.dropna().corr().iloc[0,1],3))

A positive year-over-year correlation (printed above) means the same calendar day tends to behave similarly across years, which supports keeping `lag_364`/`lag_371` (and their average, `yoy_mean`) as a long-horizon feature. It is a soft signal rather than a strong one — with about two years of data there are only 1-2 repeats of any given calendar date, so `yoy_mean` is included as one input among many rather than the main driver.

### 2.7 Autocorrelation of demand (ACF)

In [ ]:
acf = {}
for L in range(1,29):
    acf[L] = raw.groupby(["station_code","product"])[TARGET].apply(lambda s: s.autocorr(L)).mean()
acf = pd.Series(acf)
ax = acf.plot.bar(color=["#0a9c4a" if l%7==0 else "#9fb0c0" for l in acf.index], figsize=(11,3.2))
ax.set(title="Mean autocorrelation by lag (green = multiples of 7)", xlabel="lag (days)", ylabel="autocorr"); plt.tight_layout(); plt.show()
print("Top lags by autocorrelation:", list(acf.sort_values(ascending=False).head(5).round(3).items()))

Autocorrelation **peaks sharply at lag 7, 14, 21, 28 ...** (highlighted in green) and decays in between, which is the quantitative confirmation of the weekday pattern seen above. This is exactly why the feature set is built around weekly-multiple lags (`lag_7` ... `lag_371`) instead of consecutive daily lags (`lag_1`, `lag_2`, ...): the weekly lags carry most of the recoverable signal, and — more importantly — daily lags would violate the 7-day weekly-planning horizon used throughout Section 3.

### 2.8 Intermittency check by volume segment

In [ ]:
pm = raw.groupby(["station_code","product"])[TARGET].transform("mean")
seg = pd.cut(pm,[-1,np.quantile(pm,.33),np.quantile(pm,.66),1e18],labels=["low","medium","high"])
tab = raw.assign(seg=seg).groupby("seg")[TARGET].agg(mean="mean", pct_zero=lambda s:(s==0).mean()*100, n="size").round(2)
print(tab.to_string())

Even the low segment has **almost no zero days** → demand is **not intermittent**. Croston/TSB
methods are therefore inappropriate; the low segment is simply small-volume demand with relatively high
variance.

### 2.9 Fuel price is national and uniform

In [ ]:
u = raw.groupby(["country_code","date"])["fuelprice_euro95_eur_l"].nunique()
print("Max distinct euro95 prices within a (country, day):", int(u.max()), "-> uniform across stations" if u.max()==1 else "")
for c,col in [("NL","#e11d44"),("BE","#2563eb")]:
    s=raw[raw.country_code==c].groupby("date")["fuelprice_euro95_eur_l"].first()
    plt.plot(s.index,s.values,label=c,color=col,lw=1.2)
plt.title("Euro95 price by country over time (one value per country per day)"); plt.ylabel("EUR/L"); plt.legend(); plt.tight_layout(); plt.show()

Because the price series only varies **by country and by day**, not by station, it cannot explain why one station in the same country sells more than a neighbouring one on the same day — the very thing the model needs to predict. This is the evidence behind excluding the price feature group from the production configuration in Section 5 (it re-appears there only as a documented ablation, not as part of the selected model).

### 2.10 Public-holiday and school-holiday effects

In [ ]:
def cmp(flag,lab):
    a=raw[raw[flag]==1][TARGET].mean(); b=raw[raw[flag]==0][TARGET].mean()
    print(f"{lab:18s}: holiday={a:7.1f}  non-holiday={b:7.1f}  ratio={a/b:.2f}")
cmp("is_public_holiday","Public holiday"); cmp("is_school_holiday","School holiday")

Demand on holidays/school breaks **differs noticeably** from ordinary days — a travel-behaviour signal
that lags alone do not fully capture. This supports retaining the **school-holiday** group (the only
external group that proves useful).

### 2.11 Geographic map & Station × Product heatmap

In [ ]:
tot = raw.groupby("station_code").agg(lat=("lat","first"),lon=("lon","first"),demand=(TARGET,"sum")).reset_index().merge(geo[["station_code","brand"]],on="station_code",how="left")
piv = raw.pivot_table(index="station_code",columns="product",values=TARGET,aggfunc="mean").reindex(columns=["P1","P3","P2","P4","P5"])
piv = piv.loc[piv.sum(axis=1).sort_values(ascending=False).index]
import matplotlib.colors as mc
fig,ax=plt.subplots(1,2,figsize=(12,5.4),gridspec_kw={"width_ratios":[1,1]})
for brand,c in [("Brand A","#e11d44"),("Brand B","#2563eb")]:
    s=tot[tot.brand==brand]; ax[0].scatter(s.lon,s.lat,s=(s.demand/tot.demand.max()*1400)+40,c=c,alpha=.55,edgecolors="black",linewidths=.6,label=f"{brand} ({len(s)})")
for _,r in tot.iterrows(): ax[0].annotate(r.station_code,(r.lon,r.lat),fontsize=7,ha="center",va="center")
ax[0].set(title="Station map (size = volume)",xlabel="Longitude",ylabel="Latitude"); ax[0].legend()
im=ax[1].imshow(np.ma.masked_invalid(piv.values),aspect="auto",cmap="YlOrRd",norm=mc.LogNorm(vmin=np.nanmin(piv.values[piv.values>0]),vmax=np.nanmax(piv.values)))
ax[1].set_xticks(range(5)); ax[1].set_xticklabels([f"{c}\n{PRODUCT_FUEL[c]}" for c in piv.columns],fontsize=7)
ax[1].set_yticks(range(len(piv))); ax[1].set_yticklabels(piv.index,fontsize=7); ax[1].set_title("Mean litres/day — Station x Product")
plt.tight_layout(); plt.show()

The map shows a **mixed station network with the largest markers concentrated on a few sites**, and the heatmap shows that even within the same station, product mix varies (some stations barely sell certain minor fuels). Both patterns motivate the geography feature group (`G_STAT`: station/product identity, country, area type, road/building density) introduced in the next section — the model needs station-level context, not just historical demand, to explain these structural differences.

## 3. Feature engineering and explicit missing-data policy
ROVER plans **weekly**, so every historical demand feature uses information that is at least 7 calendar
days old. Before calculating those features, we create one calendar row per active station-product-day.
A product gap becomes zero only when another product proves that the same station-day was observed. If
the entire station-day is absent, demand stays unknown: that row preserves calendar spacing but is never
used as a training or evaluation target. A station-month enters modelling only when at least 28 distinct
station-days were observed. Missing historical features remain `NaN`; XGBoost learns their default split
direction, aided by lag-missing flags and rolling valid-count features.

In [ ]:
panel_df, cat_maps = build_features(raw)
status_counts = panel_df["target_status"].value_counts().rename_axis("target_status").reset_index(name="rows")
month_audit = panel_df[["station_code","year_month","station_month_observed_days","month_eligible"]].drop_duplicates()
excluded_months = month_audit.loc[~month_audit["month_eligible"]].sort_values(["year_month","station_code"])
df = panel_df.loc[panel_df["target_known"] & panel_df["month_eligible"]].reset_index(drop=True)
CORE = G_TS + G_CAL + G_STAT
G_TRAFFIC = [c for c in ["traffic_n_sensors_5km","traffic_n_sensors_10km","traffic_nearest_distance_km"] if c in df.columns]
G_WEATHER = [c for c in ["temperature","precipitation"] if c in df.columns]
G_PRICE   = [c for c in df.columns if c.startswith("price_") or c.startswith("natl_gap")]
for c in CORE + G_SCHOOL + G_TRAFFIC + G_WEATHER + G_PRICE:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
status_counts.to_csv(OUTPUT_DIR / "data_quality_target_status.csv", index=False)
excluded_months.to_csv(OUTPUT_DIR / "excluded_station_months_under_28_days.csv", index=False)
print("Calendar panel shape:", panel_df.shape, "| eligible known-target rows:", len(df))
print("Target status:", dict(zip(status_counts.target_status, status_counts.rows)))
print("Station-months excluded by <28 observed days:", len(excluded_months), "of", len(month_audit),
      f"({len(excluded_months)/len(month_audit)*100:.2f}%)")
print("Eligible rows with at least one missing model feature:", int(df[BEST_FEATURES].isna().any(axis=1).sum()))
print("BEST features:", len(BEST_FEATURES))
for nm,grp in [("Time series",G_TS),("Calendar",G_CAL),("Geography",G_STAT),("School holidays",G_SCHOOL),
               ("(extra) Weather",G_WEATHER),("(extra) Traffic",G_TRAFFIC),("(extra) Price",G_PRICE)]:
    print(f"  {nm:18s} ({len(grp):2d})")
df[["date","station_code","product","target_status","station_month_observed_days","lag_7","lag_28","lag_364","roll4_valid_count","is_school_holiday"]].head()

The model table above contains only known targets from station-months with at least 28 observed station-days. The 28-day rule is a data-quality threshold, not a claim that every lag is present: a calendar lag may still be `NaN` after a whole-station outage. Such rows remain usable because XGBoost handles missing feature values natively. By contrast, an unknown target can never teach or score the model and is excluded. The current-day audit fields (`target_known`, `target_status`, and `source_observed`) are deliberately not included in `BEST_FEATURES`, preventing target-availability leakage.

## 4. Evaluation design (chronological backtest)
We use **5 chronological folds** (test months Oct 2025 → Mar 2026); each fold trains on the past with a
28-day validation window for early stopping. Train, validation and test labels are always known targets
from eligible station-months, while their feature matrices may contain `NaN`. For the assignment's pure
`lag_7` baseline, paired model-vs-baseline metrics use only test rows where `lag_7` exists; full-coverage
model performance is reported separately. The primary metric is **WAPE**, alongside MAE and RMSE.

In [ ]:
FOLDS=[]
for s,e in [("2025-10-01","2025-11-01"),("2025-11-01","2025-12-01"),("2025-12-01","2026-01-01"),
            ("2026-01-01","2026-02-01"),("2026-02-01","2026-04-02")]:
    f=dict(test_start=pd.Timestamp(s),test_end=pd.Timestamp(e)); f["val_start"]=f["test_start"]-pd.Timedelta(days=28); FOLDS.append(f)
def wape(y,p): y,p=np.asarray(y,float),np.asarray(p,float); d=np.abs(y).sum(); return np.nan if d==0 else np.abs(y-p).sum()/d*100
from sklearn.metrics import mean_absolute_error, mean_squared_error
print(len(FOLDS),"folds; test", FOLDS[0]["test_start"].date(),"->",FOLDS[-1]["test_end"].date())
ENS1=[dict(c) for c in ENSEMBLE_CFGS]  # 1 seed x 2 objectives for a reproducible notebook runtime
def fit_predict(tr,va,te,feats,cfgs=ENS1,seeds=(SEED,)):
    ps=[]
    for sd in seeds:
        for cfg in cfgs:
            m=xgb.XGBRegressor(**{**cfg,"random_state":sd}); m.fit(tr[feats],tr[TARGET],eval_set=[(va[feats],va[TARGET])],verbose=False)
            ps.append(m.predict(te[feats]))
    return np.clip(np.mean(ps,axis=0),0,None)

The ensemble deliberately mixes **two objectives**: a Tweedie loss (`tweedie_variance_power=1.4`), designed for non-negative, right-skewed targets like litres sold, and an absolute-error loss, which is more robust to the demand spikes seen in Section 2.2. Averaging their predictions is a lightweight hedge against either objective's individual weak spot. Version 5 uses one fixed, declared seed per objective so the notebook and report are exactly reproducible. A future production service may evaluate a multi-seed ensemble, but no unexecuted multi-seed result is claimed here.

## 5. Feature-group ablation (full 5 folds)
The central question of the brief: *does external data help?* We start from the **core** set (time
series + calendar + geography) and add one group at a time. Every reported Version 5 number uses the
same fixed seed and early-stopping policy; multi-seed ensembling remains a future robustness check.

In [ ]:
SETS = {"naive_lag7":None, "core":CORE, "core+school (BEST)":CORE+G_SCHOOL,
        "core+school+weather":CORE+G_SCHOOL+G_WEATHER, "core+school+traffic":CORE+G_SCHOOL+G_TRAFFIC,
        "core+school+price":CORE+G_SCHOOL+list(G_PRICE)}
rows=[]; fold_rows=[]; store={}; best_full_coverage=[]
for name,feats in SETS.items():
    per=[]; ap=[]; paired_rows=0; known_rows=0
    for fold_id,f in enumerate(FOLDS, start=1):
        tr=df[df.date<f["val_start"]]; va=df[(df.date>=f["val_start"])&(df.date<f["test_start"])]; te=df[(df.date>=f["test_start"])&(df.date<f["test_end"])]
        paired_mask = te["lag_7"].notna().to_numpy(); paired = te.loc[paired_mask].copy()
        known_rows += len(te); paired_rows += len(paired)
        if feats is None:
            p = paired["lag_7"].to_numpy()
        else:
            p_all = fit_predict(tr,va,te,feats)
            p = p_all[paired_mask]
            if name == "core+school (BEST)":
                best_full_coverage.append(te[["date","station_code","product",TARGET,"target_status","station_month_observed_days","lag_7"]]
                                          .assign(model_prediction=p_all,fold=fold_id,lag_7_available=paired_mask))
        fold_wape = wape(paired[TARGET],p)
        fold_mae = mean_absolute_error(paired[TARGET],p)
        fold_rmse = np.sqrt(mean_squared_error(paired[TARGET],p))
        per.append((fold_wape,fold_mae,fold_rmse))
        fold_rows.append({"feature_set":name,"fold":fold_id,"test_start":f["test_start"],
                          "test_end_exclusive":f["test_end"],"test_rows":len(paired),
                          "WAPE":round(fold_wape,3),"MAE":round(fold_mae,1),"RMSE":round(fold_rmse,1)})
        ap.append(paired[["date","station_code","product",TARGET,"target_status"]].assign(pred=p,fold=fold_id))
    per=np.array(per); store[name]=pd.concat(ap)
    rows.append({"feature_set":name,"n_feat":(1 if feats is None else len(feats)),
                 "WAPE":round(per[:,0].mean(),3),"MAE":round(per[:,1].mean(),1),"RMSE":round(per[:,2].mean(),1),
                 "paired_test_rows":paired_rows,"paired_coverage_%":round(paired_rows/known_rows*100,2)})
bench=pd.DataFrame(rows); fold_metrics=pd.DataFrame(fold_rows)
base=bench.loc[bench.feature_set=="naive_lag7","WAPE"].iloc[0]
bench["vs_naive_%"]=((base-bench.WAPE)/base*100).round(1)
naive_folds = fold_metrics.loc[fold_metrics.feature_set.eq("naive_lag7"),
    ["fold","test_start","test_end_exclusive","test_rows","WAPE","MAE","RMSE"]].rename(
    columns={"WAPE":"naive_WAPE","MAE":"naive_MAE","RMSE":"naive_RMSE"})
best_folds = fold_metrics.loc[fold_metrics.feature_set.eq("core+school (BEST)"),
    ["fold","test_start","test_end_exclusive","test_rows","WAPE","MAE","RMSE"]].rename(
    columns={"WAPE":"model_WAPE","MAE":"model_MAE","RMSE":"model_RMSE"})
fold_comparison = naive_folds.merge(best_folds, on=["fold","test_start","test_end_exclusive","test_rows"], validate="one_to_one")
fold_comparison["WAPE_improvement_pp"] = (fold_comparison.naive_WAPE-fold_comparison.model_WAPE).round(3)
fold_comparison["WAPE_improvement_%"] = ((fold_comparison.naive_WAPE-fold_comparison.model_WAPE)/fold_comparison.naive_WAPE*100).round(1)
best_full_coverage = pd.concat(best_full_coverage, ignore_index=True)
full_coverage_summary = pd.DataFrame([{"scope":"BEST model, all eligible known targets",
    "rows":len(best_full_coverage), "WAPE":round(wape(best_full_coverage[TARGET],best_full_coverage.model_prediction),3),
    "MAE":round(mean_absolute_error(best_full_coverage[TARGET],best_full_coverage.model_prediction),1),
    "RMSE":round(np.sqrt(mean_squared_error(best_full_coverage[TARGET],best_full_coverage.model_prediction)),1)}])

# -------------------------------------------------------------------------
# CUSTOMER-FACING PREDICTION EVIDENCE (all paired out-of-fold test rows)
# -------------------------------------------------------------------------
model_pair = store["core+school (BEST)"].reset_index(drop=True).rename(columns={"pred":"model_prediction"})
naive_pair = store["naive_lag7"].reset_index(drop=True).rename(columns={"pred":"naive_prediction"})
pair_keys = ["fold","date","station_code","product",TARGET,"target_status"]
assert model_pair[pair_keys].equals(naive_pair[pair_keys]), "Model and naive test rows are not aligned"
prediction_comparison = model_pair[pair_keys + ["model_prediction"]].copy()
prediction_comparison["naive_prediction"] = naive_pair["naive_prediction"].to_numpy()
prediction_comparison = prediction_comparison.rename(columns={TARGET:"actual"})
prediction_comparison["naive_abs_error_L"] = (prediction_comparison["naive_prediction"] - prediction_comparison["actual"]).abs()
prediction_comparison["model_abs_error_L"] = (prediction_comparison["model_prediction"] - prediction_comparison["actual"]).abs()
prediction_comparison["error_reduction_L"] = prediction_comparison["naive_abs_error_L"] - prediction_comparison["model_abs_error_L"]
prediction_comparison["winner"] = np.select(
    [prediction_comparison["model_abs_error_L"] < prediction_comparison["naive_abs_error_L"],
     prediction_comparison["model_abs_error_L"] > prediction_comparison["naive_abs_error_L"]],
    ["selected_model","naive_lag7"], default="tie")
prediction_comparison = prediction_comparison[["fold","date","station_code","product","target_status","actual",
    "naive_prediction","model_prediction","naive_abs_error_L","model_abs_error_L","error_reduction_L","winner"]]

def pooled_metrics(pred_col):
    y=prediction_comparison["actual"].to_numpy(); p=prediction_comparison[pred_col].to_numpy()
    return dict(pooled_WAPE=round(wape(y,p),3), pooled_MAE=round(mean_absolute_error(y,p),1),
                pooled_RMSE=round(np.sqrt(mean_squared_error(y,p)),1),
                total_abs_error_L=round(float(np.abs(y-p).sum()),1),
                mean_bias_L=round(float(np.mean(p-y)),1))
customer_summary = pd.DataFrame([
    {"model":"Naive lag-7","paired_test_rows":len(prediction_comparison),
     "five_fold_mean_WAPE":float(bench.loc[bench.feature_set.eq("naive_lag7"),"WAPE"].iloc[0]),
     "vs_naive_improvement_%":0.0,"row_win_share_%":round((prediction_comparison.winner.eq("naive_lag7")).mean()*100,1),
     **pooled_metrics("naive_prediction")},
    {"model":"Selected core+school XGBoost","paired_test_rows":len(prediction_comparison),
     "five_fold_mean_WAPE":float(bench.loc[bench.feature_set.eq("core+school (BEST)"),"WAPE"].iloc[0]),
     "vs_naive_improvement_%":float(bench.loc[bench.feature_set.eq("core+school (BEST)"),"vs_naive_%"].iloc[0]),
     "row_win_share_%":round((prediction_comparison.winner.eq("selected_model")).mean()*100,1),
     **pooled_metrics("model_prediction")},
])

def product_quality(g):
    denom=g["actual"].abs().sum()
    return pd.Series({"rows":len(g),"actual_volume_L":round(g["actual"].sum(),1),
        "naive_WAPE":round(g["naive_abs_error_L"].sum()/denom*100,2),
        "model_WAPE":round(g["model_abs_error_L"].sum()/denom*100,2),
        "WAPE_improvement_pp":round((g["naive_abs_error_L"].sum()-g["model_abs_error_L"].sum())/denom*100,2),
        "model_row_win_share_%":round(g["winner"].eq("selected_model").mean()*100,1)})
product_comparison = prediction_comparison.groupby("product",group_keys=False).apply(product_quality).reset_index()
latest_network_snapshot = (prediction_comparison.sort_values(["date","station_code","product"])
                           .groupby(["station_code","product"],as_index=False).tail(1)
                           .sort_values(["station_code","product"]).reset_index(drop=True))
daily_network_comparison = (prediction_comparison.groupby("date",as_index=False)
    .agg(actual=("actual","sum"),naive_prediction=("naive_prediction","sum"),
         model_prediction=("model_prediction","sum")))
daily_network_comparison["naive_abs_error_L"] = (daily_network_comparison.naive_prediction-daily_network_comparison.actual).abs()
daily_network_comparison["model_abs_error_L"] = (daily_network_comparison.model_prediction-daily_network_comparison.actual).abs()

# Contract checks: the exported evidence must reproduce the headline benchmark exactly.
assert len(prediction_comparison) == int(bench.loc[bench.feature_set.eq("core+school (BEST)"),"paired_test_rows"].iloc[0])
for model_name,pred_col in [("naive_lag7","naive_prediction"),("core+school (BEST)","model_prediction")]:
    fold_mean = prediction_comparison.groupby("fold").apply(lambda g:wape(g["actual"],g[pred_col])).mean()
    expected = float(bench.loc[bench.feature_set.eq(model_name),"WAPE"].iloc[0])
    assert np.isclose(fold_mean,expected,atol=0.001), (model_name,fold_mean,expected)
assert len(fold_comparison)==len(FOLDS) and (fold_comparison.test_rows>0).all()

bench.to_csv(OUTPUT_DIR / "benchmark_ablation_results.csv", index=False)
fold_metrics.to_csv(OUTPUT_DIR / "benchmark_fold_results.csv", index=False)
fold_comparison.to_csv(OUTPUT_DIR / "best_model_fold_comparison.csv", index=False)
full_coverage_summary.to_csv(OUTPUT_DIR / "best_model_full_coverage_summary.csv", index=False)
prediction_comparison.to_csv(OUTPUT_DIR / "best_model_test_predictions.csv", index=False, float_format="%.3f")
best_full_coverage.rename(columns={TARGET:"actual"}).to_csv(OUTPUT_DIR / "best_model_full_coverage_predictions.csv", index=False, float_format="%.3f")
customer_summary.to_csv(OUTPUT_DIR / "customer_prediction_summary.csv", index=False)
product_comparison.to_csv(OUTPUT_DIR / "prediction_quality_by_product.csv", index=False)
latest_network_snapshot.to_csv(OUTPUT_DIR / "latest_network_prediction_snapshot.csv", index=False, float_format="%.3f")
daily_network_comparison.to_csv(OUTPUT_DIR / "daily_network_prediction_comparison.csv", index=False, float_format="%.3f")
print("OUTPUT SAVED:", OUTPUT_DIR / "benchmark_ablation_results.csv")
print("OUTPUT SAVED:", OUTPUT_DIR / "benchmark_fold_results.csv")
print("OUTPUT SAVED:", OUTPUT_DIR / "best_model_fold_comparison.csv")
print("OUTPUT SAVED:", OUTPUT_DIR / "best_model_full_coverage_summary.csv")
print("OUTPUT SAVED: prediction-level and customer comparison CSV files")
print("\nBENCHMARK SUMMARY (5-fold paired mean)")
print(bench.to_string(index=False))
print("\nNAIVE VS SELECTED MODEL BY FOLD")
print(fold_comparison.to_string(index=False))
print("\nCUSTOMER SUMMARY (all paired out-of-fold test rows; no cherry-picking)")
print(customer_summary.to_string(index=False))
print("\nQUALITY BY PRODUCT")
print(product_comparison.to_string(index=False))
latest_network_snapshot[["date","station_code","product","actual","naive_prediction","model_prediction","winner"]].head(20)


In [ ]:
b=bench[bench.feature_set!="naive_lag7"].set_index("feature_set")["WAPE"]
display_names={"core":"Core","core+school (BEST)":"Core + school (selected)",
               "core+school+weather":"Core + school + weather",
               "core+school+traffic":"Core + school + traffic",
               "core+school+price":"Core + school + price"}
b.index=[display_names[i] for i in b.index]
ax=b.plot.bar(color=["#0a9c4a" if "selected" in i else "#9fb0c0" for i in b.index],figsize=(9,3.4))
ax.axhline(base,ls="--",color="#e11d44",label=f"Naive lag-7 = {base:.2f}")
ax.set(title="WAPE by feature set (5-fold paired mean; lower is better)",xlabel="",ylabel="WAPE (%)"); ax.legend(); plt.xticks(rotation=20,ha="right"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_ablation_wape.png", dpi=180, bbox_inches="tight"); plt.show()
print("FIGURE SAVED:", OUTPUT_DIR / "feature_ablation_wape.png")

fig,ax=plt.subplots(figsize=(12,4.2))
ax.plot(daily_network_comparison.date,daily_network_comparison.actual,color="#111827",lw=2.0,label="Actual")
ax.plot(daily_network_comparison.date,daily_network_comparison.naive_prediction,color="#f59e0b",lw=1.2,ls="--",alpha=.9,label="Naive lag-7")
ax.plot(daily_network_comparison.date,daily_network_comparison.model_prediction,color="#2563eb",lw=1.5,alpha=.9,label="Selected model")
ax.set(title="Daily network demand: actual vs naive vs selected model (all 5 test folds)",xlabel="Date",ylabel="Demand (litres)")
ax.legend(ncol=3); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actual_vs_prediction_network.png", dpi=180, bbox_inches="tight"); plt.show()
print("FIGURE SAVED:", OUTPUT_DIR / "actual_vs_prediction_network.png")

Reading the bar chart against the naive baseline: all model-versus-baseline values use the same 11,932 paired test rows (99.87% coverage), so the comparison is not improved by silently dropping hard cases for only one method. The selected core+school model reduces paired WAPE from 25.478% to 20.153%. On all 11,947 eligible known-target test rows, including cases where lag-7 is unavailable, its separately reported full-coverage WAPE is 20.571%. For customer review, `best_model_test_predictions.csv` now exposes every paired out-of-fold row with actual demand, both predictions, both absolute errors and the row winner. `customer_prediction_summary.csv`, `prediction_quality_by_product.csv`, and `latest_network_prediction_snapshot.csv` provide progressively more compact views without selecting only favourable examples.

## 6. Statistical testing (paired bootstrap by date)
A lower mean WAPE is not necessarily a real improvement. We use a **paired bootstrap by date** (2000
resamples) on the WAPE difference between naive and the BEST model; if the confidence interval excludes
zero, the improvement is robust.

In [ ]:
yv=store["core+school (BEST)"].reset_index(drop=True)
nv=store["naive_lag7"].reset_index(drop=True)
pair_keys=["date","station_code","product"]
assert yv[pair_keys].equals(nv[pair_keys]) and np.allclose(yv[TARGET],nv[TARGET]), "Paired bootstrap rows are misaligned"
dates=yv["date"].values; uniq=np.unique(dates); rng=np.random.default_rng(42); diffs=[]
date_indices={d:np.flatnonzero(dates==d) for d in uniq}
for _ in range(2000):
    samp=rng.choice(uniq,size=len(uniq),replace=True)
    idx=np.concatenate([date_indices[d] for d in samp])
    diffs.append(wape(nv[TARGET].to_numpy()[idx],nv["pred"].to_numpy()[idx]) - wape(yv[TARGET].to_numpy()[idx],yv["pred"].to_numpy()[idx]))
diffs=np.array(diffs)
bootstrap_summary = pd.DataFrame([{"metric":"WAPE improvement (naive - BEST)",
                                   "median_pp":round(float(np.median(diffs)),2),
                                   "ci_2_5_pp":round(float(np.percentile(diffs,2.5)),2),
                                   "ci_97_5_pp":round(float(np.percentile(diffs,97.5)),2),
                                   "p_positive":round(float((diffs>0).mean()),3)}])
print(f"WAPE improvement (naive - selected): median {np.median(diffs):.2f} pp | 95% CI [{np.percentile(diffs,2.5):.2f}, {np.percentile(diffs,97.5):.2f}] | share positive={ (diffs>0).mean():.3f}")
bootstrap_summary.to_csv(OUTPUT_DIR / "bootstrap_wape_summary.csv", index=False)
print("OUTPUT SAVED:", OUTPUT_DIR / "bootstrap_wape_summary.csv")
plt.hist(diffs,bins=40,color="#0a9c4a",alpha=.85); plt.axvline(0,color="#e11d44",ls="--")
plt.title("Bootstrap distribution of WAPE improvement vs naive"); plt.xlabel("WAPE improvement (percentage points)"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bootstrap_wape_improvement.png", dpi=180, bbox_inches="tight"); plt.show()
print("FIGURE SAVED:", OUTPUT_DIR / "bootstrap_wape_improvement.png")

The bootstrap distribution sits **entirely to the right of zero**, i.e. resampling the test dates 2,000 times never produces a sample where the naive baseline beats the selected model. This directly answers the assignment's first research question — the improvement over the naive 'last week' heuristic is not a fluke of one lucky test period, it is statistically robust across time.

## 7. Error by segment and by fuel

In [ ]:
ap = store["core+school (BEST)"].copy()
pm2 = raw.groupby(["station_code","product"])[TARGET].mean()
fuel_w = ap.assign(main=ap["product"].isin(MAIN_PRODUCTS))
print("WAPE main fuels (Euro95+Diesel):", round(wape(fuel_w[fuel_w.main][TARGET],fuel_w[fuel_w.main]["pred"]),2))
print("WAPE other products (P2/P4/P5):", round(wape(fuel_w[~fuel_w.main][TARGET],fuel_w[~fuel_w.main]["pred"]),2))
byp = ap.groupby("product").apply(lambda g: wape(g[TARGET],g["pred"])).round(2)
byp.index=[f"{p} {PRODUCT_FUEL[p]}" for p in byp.index]; print("\nWAPE by product:"); print(byp.to_string())
product_wape = byp.rename("WAPE").reset_index().rename(columns={"index":"product"})
product_wape.to_csv(OUTPUT_DIR / "product_wape_summary.csv", index=False)
print("OUTPUT SAVED:", OUTPUT_DIR / "product_wape_summary.csv")

The gap between the main-fuel and minor-fuel WAPE printed above is expected, not a modelling failure: Euro95 and Diesel have thousands of litres/day of volume, so their percentage error is naturally more stable, while AdBlue/LPG sell in small, lumpy quantities where the same absolute error translates into a much larger percentage. Because Euro95+Diesel make up the large majority of delivered litres, the main-fuel number is the more relevant one for truck-loading decisions.

## 8. Quantile (safety-stock) layer & calibration
For inventory, the model additionally predicts the **quantiles** P50/P90/P95. Coverage is the share of
days on which realised demand ≤ forecast; under good calibration, coverage approximates the nominal
level.

In [ ]:
quantile_parts=[]; quantile_fold_rows=[]
for fold_id,f in enumerate(FOLDS,start=1):
    tr=df[df.date<f["val_start"]]
    va=df[(df.date>=f["val_start"])&(df.date<f["test_start"])]
    te=df[(df.date>=f["test_start"])&(df.date<f["test_end"]) & df["lag_7"].notna()].copy()  # paired with pure lag-7
    fold_preds = te[["date","station_code","product",TARGET]].rename(columns={TARGET:"actual"}).copy()
    fold_preds["fold"] = fold_id
    fold_preds["naive"] = te["lag_7"].values
    for q in QUANTILES:
        mq=xgb.XGBRegressor(**QUANTILE_PARAMS,quantile_alpha=q,random_state=SEED)
        mq.fit(tr[BEST_FEATURES],tr[TARGET],eval_set=[(va[BEST_FEATURES],va[TARGET])],verbose=False)
        pq=np.clip(mq.predict(te[BEST_FEATURES]),0,None)
        qname=f"P{int(q*100)}"; fold_preds[f"pred_p{int(q*100)}"] = pq
        coverage=np.mean(te[TARGET].values<=pq)*100
        quantile_fold_rows.append({"fold":fold_id,"test_start":f["test_start"],"test_end_exclusive":f["test_end"],
            "Quantile":qname,"rows":len(te),"Coverage %":round(coverage,1),"Stockout %":round(100-coverage,1)})
    quantile_parts.append(fold_preds)
quantile_preds=pd.concat(quantile_parts,ignore_index=True)
quantile_calibration_by_fold=pd.DataFrame(quantile_fold_rows)
cov=[]
for q in QUANTILES:
    qname=f"P{int(q*100)}"; pred_col=f"pred_p{int(q*100)}"
    coverage=np.mean(quantile_preds.actual.values<=quantile_preds[pred_col].values)*100
    cov.append({"Quantile":qname,"rows":len(quantile_preds),"Coverage %":round(coverage,1),"Stockout %":round(100-coverage,1)})
covdf=pd.DataFrame(cov)
assert quantile_calibration_by_fold.groupby("Quantile")["fold"].nunique().eq(len(FOLDS)).all()
ax=covdf.plot.bar(x="Quantile",y="Coverage %",color="#2563eb",legend=False,figsize=(6,3.2))
for q in QUANTILES: ax.axhline(q*100,ls="--",lw=.7,color="#888")
ax.set(title="Quantile calibration across all 5 folds (dashed = nominal)",ylabel="coverage %"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "quantile_calibration.png", dpi=180, bbox_inches="tight"); plt.show()
covdf.to_csv(OUTPUT_DIR / "quantile_calibration.csv", index=False)
quantile_calibration_by_fold.to_csv(OUTPUT_DIR / "quantile_calibration_by_fold.csv", index=False)
quantile_preds.to_csv(OUTPUT_DIR / "quantile_preds_for_simulation.csv", index=False)
print("OUTPUT SAVED:", OUTPUT_DIR / "quantile_calibration.csv")
print("OUTPUT SAVED:", OUTPUT_DIR / "quantile_calibration_by_fold.csv")
print("OUTPUT SAVED:", OUTPUT_DIR / "quantile_preds_for_simulation.csv")
print("FIGURE SAVED:", OUTPUT_DIR / "quantile_calibration.png")
print("\nPOOLED QUANTILE CALIBRATION (all 5 folds)")
print(covdf.to_string(index=False))
print("\nQUANTILE CALIBRATION BY FOLD")
print(quantile_calibration_by_fold.to_string(index=False))
covdf


Coverage close to the nominal quantile level (e.g. P95 coverage near 95%) means the model is **well-calibrated**, not just accurate on average. Version 5 reports both pooled calibration and each of the same five chronological folds used for point-model evaluation. Section 9 tests whether using that safety-stock level, instead of the plain point forecast, actually reduces stockouts in a capacity-capped simulation across all five folds.

## 9. Levels-penalty simulation
The ultimate objective is to reduce inventory cost, not only WAPE. WAPE treats over- and under-forecast
equally, but the real cost is **asymmetric** (a shortage costs far more than a surplus).

**Revision note:** an earlier version of this section read a precomputed `sim_penalty.csv` table that treated "overfill" as an unbounded bookkeeping cost. In review, Quirijn Schevenhoven correctly pointed out that this is not physically possible -- a tank cannot hold more fuel than its capacity, and any excess the truck cannot unload has to be rerouted or replanned, which the old table did not represent at all. This section replaces that table with a simulation that **runs in this notebook** and caps every delivery at a tank-capacity proxy, and reports the volume that could not be delivered (`undeliverable_L`) as a distinct, explicit cost.

In [ ]:
# Capacity proxy: Cofano's brief lists station tank capacities as available Master Data, but no such file
# was provided to this team. Until real capacities are supplied, each station-product tank capacity is
# proxied as CAPACITY_MULTIPLIER x the heaviest 7-day volume that pair ever sold BEFORE the test fold
# (no leakage). The sweep further down shows the conclusion is not an artifact of one lucky multiplier.
CAPACITY_MULTIPLIER = 1.3
STOCKOUT_WEIGHT, CARRY_WEIGHT, REPLAN_WEIGHT = 10, 1, 5  # synthetic proxy weights, not measured costs

def peak_complete_calendar_week(g):
    """Maximum demand over seven consecutive calendar days with seven known targets."""
    g = g.sort_values("date")
    gaps = g["date"].diff().dropna()
    if not gaps.eq(pd.Timedelta(days=1)).all():
        raise AssertionError("Capacity history must be a continuous calendar-day panel.")
    return g[TARGET].rolling(7, min_periods=7).sum().max()

peak_weekly_by_fold = {}
capacity_proxy_audit = []
for fold_id,f in enumerate(FOLDS,start=1):
    hist = panel_df[panel_df.date < f["test_start"]].sort_values(["station_code","product","date"])
    peak = hist.groupby(["station_code","product"], group_keys=False).apply(peak_complete_calendar_week)
    peak_weekly_by_fold[fold_id] = peak
    capacity_proxy_audit.append(peak.rename("peak_complete_7_calendar_day_demand_L").reset_index().assign(fold=fold_id))
capacity_proxy_audit = pd.concat(capacity_proxy_audit, ignore_index=True)
capacity_proxy_audit.to_csv(OUTPUT_DIR / "capacity_proxy_by_fold.csv", index=False)
assert capacity_proxy_audit.groupby("fold").size().eq(df[["station_code","product"]].drop_duplicates().shape[0]).all()
print("OUTPUT SAVED:", OUTPUT_DIR / "capacity_proxy_by_fold.csv")

def simulate_policy(qp, forecast_col, capacity_multiplier):
    """Weekly order-up-to simulation, delivery capped at a tank-capacity proxy."""
    rows = []
    for (fold_id, station, product), g in qp.groupby(["fold","station_code","product"]):
        g = g.sort_values("date")
        peak_weekly = peak_weekly_by_fold[int(fold_id)].get((station, product), np.nan)
        capacity = peak_weekly * capacity_multiplier
        if not capacity or not np.isfinite(capacity) or capacity <= 0:
            capacity = max(g["actual"].max()*2, 1.0)
        g = g.assign(iso_week=g["date"].dt.isocalendar().week.astype(str)+"-"+g["date"].dt.isocalendar().year.astype(str))
        stock = capacity / 2  # assumed starting stock for the first simulated week only
        for wk, wg in g.groupby("iso_week", sort=False):
            weekly_forecast = wg[forecast_col].clip(lower=0).sum()
            desired = max(0.0, weekly_forecast - stock)
            delivered = min(desired, max(0.0, capacity - stock))       # <-- the physical capacity cap
            undeliverable = desired - delivered                        # <-- volume that needs a reroute/replan
            stock += delivered
            shortfall, stockout_days = 0.0, 0
            for demand in wg["actual"].clip(lower=0):
                if demand > stock:
                    shortfall += demand - stock; stockout_days += 1; stock = 0.0
                else:
                    stock -= demand
            rows.append(dict(shortfall_L=shortfall, stockout_days=stockout_days,
                              undeliverable_L=undeliverable, end_stock_L=stock))
    out = pd.DataFrame(rows)
    s = dict(stockout_days=int(out.stockout_days.sum()), shortfall_L=round(out.shortfall_L.sum(),1),
             feasible_carryover_L=round(out.end_stock_L.sum(),1), undeliverable_L=round(out.undeliverable_L.sum(),1),
             replan_events=int((out.undeliverable_L > 1e-6).sum()))
    s["penalty"] = round(STOCKOUT_WEIGHT*s["shortfall_L"] + CARRY_WEIGHT*s["feasible_carryover_L"]
                          + REPLAN_WEIGHT*s["undeliverable_L"], 1)
    return s

POLICIES = {"naive": "naive", "model_p50": "pred_p50", "model_p95": "pred_p95"}
pen_rows = []
for name, col in POLICIES.items():
    s = simulate_policy(quantile_preds, col, CAPACITY_MULTIPLIER); s["policy"] = name; pen_rows.append(s)
pen = pd.DataFrame(pen_rows).set_index("policy")
base = pen.loc["naive","penalty"]; pen["penalty_vs_naive_%"] = ((base - pen["penalty"]) / base * 100).round(1)
pen = pen.reset_index()
pen_by_fold_rows=[]
for fold_id in range(1,len(FOLDS)+1):
    fold_qp=quantile_preds.loc[quantile_preds.fold.eq(fold_id)]
    for name,col in POLICIES.items():
        s=simulate_policy(fold_qp,col,CAPACITY_MULTIPLIER); s.update({"fold":fold_id,"policy":name}); pen_by_fold_rows.append(s)
pen_by_fold=pd.DataFrame(pen_by_fold_rows)
fold_naive_penalty=pen_by_fold.loc[pen_by_fold.policy.eq("naive"),["fold","penalty"]].rename(columns={"penalty":"naive_penalty"})
pen_by_fold=pen_by_fold.merge(fold_naive_penalty,on="fold",validate="many_to_one")
pen_by_fold["penalty_vs_naive_%"]=((pen_by_fold.naive_penalty-pen_by_fold.penalty)/pen_by_fold.naive_penalty*100).round(1)
assert pen_by_fold.groupby("policy")["fold"].nunique().eq(len(FOLDS)).all()
pen.to_csv(OUTPUT_DIR / "sim_penalty_capacity_aware.csv", index=False)
pen_by_fold.to_csv(OUTPUT_DIR / "sim_penalty_by_fold.csv", index=False)
print("OUTPUT SAVED:", OUTPUT_DIR / "sim_penalty_capacity_aware.csv")
print("OUTPUT SAVED:", OUTPUT_DIR / "sim_penalty_by_fold.csv")
print(f"SIMULATION SCOPE: all {len(FOLDS)} folds | capacity proxy multiplier={CAPACITY_MULTIPLIER} | penalty weights stockout/carry/replan={STOCKOUT_WEIGHT}/{CARRY_WEIGHT}/{REPLAN_WEIGHT}")

fig,ax=plt.subplots(1,2,figsize=(11,3.4))
policy_order=["naive","model_p50","model_p95"]; policy_labels=["Naive lag-7","P50 quantile","P95 quantile"]
plot_pen=pen.set_index("policy").loc[policy_order]
ax[0].bar(policy_labels, plot_pen.stockout_days, color="#e11d65"); ax[0].set(title="Station-product stockout-day events", ylabel="Events")
ax[1].bar(policy_labels, plot_pen.penalty/1e6, color="#2563eb"); ax[1].set(title="Weighted proxy penalty (millions)", ylabel="Weighted proxy penalty (M)")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "capacity_simulation.png", dpi=180, bbox_inches="tight"); plt.show()
print("FIGURE SAVED:", OUTPUT_DIR / "capacity_simulation.png")
print("Penalty vs naive (%):", pen.set_index("policy")["penalty_vs_naive_%"].round(1).to_dict())
print("\nCAPACITY-CAPPED SIMULATION BY FOLD")
print(pen_by_fold.to_string(index=False))
pen


In [ ]:
# Sensitivity check: does the conclusion depend on the exact capacity-multiplier assumption?
sweep_rows = []
for mult in [1.1, 1.3, 1.5, 2.0, 3.0]:
    for name, col in POLICIES.items():
        s = simulate_policy(quantile_preds, col, mult); s["policy"] = name; s["capacity_multiplier"] = mult
        sweep_rows.append(s)
sweep = pd.DataFrame(sweep_rows)
sweep_pivot = sweep.pivot(index="capacity_multiplier", columns="policy", values="penalty")
sweep_pivot["p95_vs_naive_%"] = ((sweep_pivot["naive"] - sweep_pivot["model_p95"]) / sweep_pivot["naive"] * 100).round(1)
sweep_pivot.to_csv(OUTPUT_DIR / "sim_capacity_sensitivity.csv")
print("OUTPUT SAVED:", OUTPUT_DIR / "sim_capacity_sensitivity.csv")
sweep_pivot.round(1)


**Reading the five-fold capacity-capped result:** at the primary 1.3x capacity proxy, pooled P95 reduces simulated station-product stockout-day events from 1,401 to 29 and lowers the aggregate weighted proxy penalty by 14.6% versus naive. The proxy is now based on each pair's maximum complete seven-calendar-day demand window before the test fold; unknown targets are not converted to zero. Across capacity multipliers from 1.1x to 3.0x, the aggregate improvement ranges from 7.2% to 14.6%. The fold table is deliberately retained: P95 is not uniformly better in every period (Fold 5 is 3.5% worse at 1.3x), and P50 is 23.9% worse in aggregate. This confirms the review comment that planning to a high quantile is not free. The final cost/service choice still needs actual tank-capacity and replanning-cost data.

## 10. Conclusions

This notebook answers the five deliverables from the assignment brief directly against the data used above:

1. **Data analysis** (Section 2): demand is right-skewed, dominated by Euro95/Diesel, has a strong weekly
   cycle (confirmed by the ACF peaks at multiples of 7) and a smaller school-holiday effect; annual
   seasonality is only weakly testable with about two years of history.
2. **Model development** (Sections 3-5): a leakage-safe, weekly-horizon feature set feeds a two-objective
   (Tweedie + MAE) XGBoost ensemble, backtested over 5 chronological folds. Version 5 explicitly separates
   observed targets, conditional zeros, unknown station-product-day targets arising from whole-station-day gaps, and missing historical features.
3. **Benchmarking** (Sections 5-6): the selected `core+school` configuration cuts WAPE from the naive lag-7
   baseline by roughly a fifth, and a 2,000-resample paired bootstrap shows the improvement is statistically
   robust (the confidence interval excludes zero).
4. **Feature engineering** (Section 5): static station/geography context remains part of the core model.
   School-holiday variables are the only additional time-varying external driver group retained in Version 5;
   their 0.143-point incremental WAPE gain was not separately significance-tested. Weather and dynamic traffic
   variables do not outperform the selected configuration, while the price group performs materially worse.
5. **The framework / blueprint**: Section 8 evaluates P50/P90/P95 on all five chronological folds; pooled
   coverage is 54.2%/90.7%/94.7%. Section 9 applies the same five-fold scope to a capacity-capped
   order-up-to stress test. At the 1.3x complete-calendar-window proxy, pooled P95 reduces simulated
   station-product stockout-day events from 1,401 to 29 and the aggregate weighted proxy penalty by 14.6%;
   the sensitivity sweep remains positive from 7.2% to 14.6%. However,
   the fold-level evidence is not uniformly positive and P50 is worse than naive, so "just selecting P95"
   remains too simplistic. The production decision still requires real per-station tank capacities and
   replanning costs rather than the historical-volume proxy used here.

**Current scope and honest limitations:** this notebook is a single, self-contained research prototype, not
yet a modular production service. Tank capacity is only a *proxy* (1.3x maximum complete seven-calendar-day
historical demand,
swept from 1.1x-3.0x for robustness) because Cofano has not yet supplied real per-station tank-capacity
data, despite it being listed as available Master Data in the brief -- obtaining that data is now the
single highest-value next step for this simulation. Truck/route constraints and supplier delivery rules
are also not modelled -- Section 9 remains a stress test of the *forecast*, not a proposed scheduling rule.
Turning this into a production pipeline (separate feature-building, training, evaluation and prediction
steps, run on a schedule and monitored for drift) is future work, not something this notebook already
implements.